# Fine-tuning example

Compare TabPFN-2.5 before and after one epoch of fine-tuning on Churn, using two
preprocessing options. Run the cells in order. The training cell starts the runs.


## 1. Load the project

Complete the [setup](../README.md#setup) and select the project's `.venv` kernel.
The cell below loads the local `.env` and locates the results directory.


In [ ]:
import json
import os
import sys

import pandas as pd
import yaml
from dotenv import load_dotenv
from IPython.display import Markdown, display

from experiments.configuration import expand_runs
from experiments.notebook import run_command
from utils.paths import get_paths, get_repo_root

root = get_repo_root()
load_dotenv(root / ".env", override=False)
results = (root / get_paths().results_root).resolve()


## 2. Choose settings

The example runs both `native` (model-specific) and `shared` preprocessing on the
same split. Change the dataset, seed or training budget below.
`context_size` is the number of rows per training chunk.


In [ ]:
config = yaml.safe_load((root / "configs/examples/introductory.yaml").read_text())
config["datasets"] = ["churn"]
config["split"]["seeds"] = [0]
config["settings"]["foundation_model.epochs"] = 1
config["settings"]["foundation_model.context_size"] = 1024
config["parameters"]["preprocessing"]["values"] = ["native", "shared"]


Optional tracking: keep `disabled` for local reports, choose `offline` for local
W&B logs, or `online` to upload them. Defaults come from `.env` where credentials can be set safely.


In [ ]:
mode = os.environ.get("WANDB_MODE", "disabled")
project = os.environ.get("WANDB_PROJECT", "tfm-finetuning")
entity = os.environ.get("WANDB_ENTITY", "")  # optional account/team

if mode not in {"online", "offline", "disabled"}:
    raise ValueError("Choose online, offline, or disabled.")
os.environ.update(WANDB_MODE=mode, WANDB_PROJECT=project)
config["wandb"] = {"project": project, "mode": mode}
if entity:
    os.environ["WANDB_ENTITY"] = entity
    config["wandb"]["entity"] = entity
else:
    os.environ.pop("WANDB_ENTITY", None)
print(f"Tracking: {mode}, project: {project}")


## 3. Preview

Save the selected settings to `results/notebook/introductory.yaml` and list the planned runs.
The training cell reads this file. The defaults produce two runs.
No data or weights are downloaded yet.

Alternatively, run the saved configuration from a terminal at the project root:

```bash
uv run tfm-experiment --config results/notebook/introductory.yaml
```

If `RESULTS_ROOT` is configured, use the saved configuration path printed below.


In [ ]:
config_path = results / "notebook" / "introductory.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
command = [
    sys.executable, "-m", "experiments.run_experiment",
    "--config", str(config_path), "--results-root", str(results),
]
print(f"Saved configuration: {config_path}")


In [ ]:
runs = expand_runs(config, repo_root=root)
display(pd.DataFrame([
    {key: run[key] for key in ("dataset", "model", "preprocessing", "fold", "seed")}
    for run in runs
]))


## 4. Prepare weights and train

This cell downloads or verifies the TabPFN-2.5 weights. It requires Hugging Face model
access via `HF_TOKEN` or a saved login. Missing datasets download automatically during
training.


In [ ]:
run_command([
    sys.executable, "-m", "data.download_tabpfn_model_checkpoints", "--group", "v2.5",
])


Run both variants sequentially. Output streams below. Online tracking prompts for
W&B login if needed. Completed runs are skipped when rerun. One epoch may not improve
the original model.


In [ ]:
if mode == "online":
    import wandb

    if not wandb.login():
        raise RuntimeError("Authenticate with W&B or choose offline/disabled tracking.")
run_command(command)


## 5. Compare results

Compare the original model, the validation-selected checkpoint and the final checkpoint.
Selected epoch `0` means validation preferred the original model. For this example,
higher ROC-AUC is better. Use validation to choose settings and test data to report results.


In [ ]:
records = []
for run in runs:
    folder = results / run["study"] / run["run_id"]
    state_path = folder / "run.json"
    report_path = folder / "finetune_report.json"
    if not state_path.is_file() or not report_path.is_file():
        raise RuntimeError(f"Run training first. Results are missing in {folder}.")
    if json.loads(state_path.read_text())["status"] != "complete":
        raise RuntimeError(f"This run has not completed. Inspect {state_path}.")
    report = json.loads(report_path.read_text())
    metric = report["score_metric"]
    for split in ("val", "test"):
        records.append({
            "preprocessing": run["preprocessing"],
            "seed": run["seed"],
            "split": split,
            "metric": metric,
            "ICL": report["zero_shot_metrics"][split][metric],
            "validation-selected": report["selected_checkpoint_metrics"][split][metric],
            "last checkpoint": report["final_checkpoint_metrics"][split][metric],
            "selected epoch": report["evaluation"]["selected_epoch"],
        })
    if report.get("wandb", {}).get("url"):
        display(Markdown(f"[W&B run]({report['wandb']['url']})"))
comparison = pd.DataFrame(records)
display(comparison.round(4))


For the full thesis experiments, continue with [HYDRA experiments](hydra_experiments.ipynb).
